<a href="https://colab.research.google.com/github/pufferfish305206-sol/HUIT-TimeSeries-FastFood/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# CODE CHUẨN HOÀN CHỈNH CHO TV4 (CHẠY TOÀN BỘ TRONG 1 CELL)
# ==============================================================================

# 1. CÀI ĐẶT VÀ KHAI BÁO THƯ VIỆN
if (!require("tidyverse")) install.packages("tidyverse", quiet = TRUE)
if (!require("lubridate")) install.packages("lubridate", quiet = TRUE)

library(tidyverse)
library(lubridate)

# 2. TẢI VÀ LƯU DATASET GỐC (data_goc.csv)
file_name <- "/content/delivery_data_with_restaurants.csv"

# Kiểm tra sự tồn tại của file
if (!file.exists(file_name)) {
  stop("⚠️ LỖI: Chưa thấy file 'delivery_data_with_restaurants.csv' trên Colab! Bạn hãy upload file này vào mục Files ở cột bên trái trước nhé.")
}

df <- read_csv(file_name, show_col_types = FALSE)

# Xuất file data_goc.csv
write_csv(df, "data_goc.csv")
cat("✓ Đã xuất thành công file: data_goc.csv\n\n")

# 3. KIỂM TRA & LÀM SẠCH DỮ LIỆU
cat("--- CẤU TRÚC DỮ LIỆU GỐC ---\n")
glimpse(df)

missing_counts <- colSums(is.na(df))
num_duplicates <- sum(duplicated(df))

# Tự động tìm cột chứa ngày/thời gian
date_col <- names(df)[grep("date|time|ngay|created", names(df), ignore.case = TRUE)][1]
if (is.na(date_col)) date_col <- names(df)[1]

cat("\nTự động xác định cột ngày tháng:", date_col, "\n")

df_cleaned <- df %>%
  distinct() %>% # Xóa trùng lặp
  drop_na(all_of(date_col)) %>% # Loại bỏ NA ở cột ngày
  mutate(
    order_date_clean = as_date(parse_date_time(!!sym(date_col), orders = c("ymd", "dmy", "mdy", "ymd HMS", "dmy HMS")))
  ) %>%
  filter(!is.na(order_date_clean)) %>%
  arrange(order_date_clean)

# Xuất cleaned_data.csv
write_csv(df_cleaned, "cleaned_data.csv")
cat("✓ Đã xuất thành công file: cleaned_data.csv\n\n")

# 4. TỔNG HỢP SỐ LƯỢNG ĐƠN HÀNG THEO NGÀY (DAILY ORDERS)
daily_orders <- df_cleaned %>%
  group_by(date = order_date_clean) %>%
  summarise(
    total_orders = n(),
    .groups = 'drop'
  )

# Điền bổ sung các ngày bị thiếu trong chuỗi thời gian (nếu có)
all_dates <- tibble(date = seq(min(daily_orders$date), max(daily_orders$date), by = "day"))
daily_orders <- all_dates %>%
  left_join(daily_orders, by = "date") %>%
  mutate(total_orders = replace_na(total_orders, 0)) %>%
  arrange(date)

# Xuất daily_orders.csv
write_csv(daily_orders, "daily_orders.csv")
cat("✓ Đã xuất thành công file: daily_orders.csv\n\n")

# 5. IN NHẬT KÝ XỬ LÝ DỮ LIỆU (DATA LOG)
cat("==============================================================================\n")
cat("                        NHẬT KÝ XỬ LÝ DỮ LIỆU (DATA LOG)                      \n")
cat("==============================================================================\n")
cat(" - Số lượng dòng dữ liệu gốc          :", nrow(df), "dòng\n")
cat(" - Số lượng bản ghi trùng lặp đã xóa  :", num_duplicates, "dòng\n")
cat(" - Tổng số giá trị khuyết (NA)        :", sum(missing_counts), "giá trị\n")
cat(" - Số lượng dòng dữ liệu sạch         :", nrow(df_cleaned), "dòng\n")
cat(" - Ngày bắt đầu chuỗi thời gian       :", as.character(min(daily_orders$date)), "\n")
cat(" - Ngày kết thúc chuỗi thời gian      :", as.character(max(daily_orders$date)), "\n")
cat(" - Tổng số ngày theo dõi              :", nrow(daily_orders), "ngày\n")
cat(" - Các sản phẩm đã xuất thành công    : data_goc.csv, cleaned_data.csv, daily_orders.csv\n")
cat("==============================================================================\n")

✓ Đã xuất thành công file: data_goc.csv

--- CẤU TRÚC DỮ LIỆU GỐC ---
Rows: 10,000
Columns: 12
$ order_id              <chr> "e5b99ed3-4bd6-4aed-9583-989823be022d", "fd83af0…
$ order_time            <dttm> 2025-04-10 13:21:00, 2025-01-26 08:45:00, 2025-…
$ delivery_time         <dttm> 2025-04-10 13:58:00, 2025-01-26 09:06:00, 2025-…
$ delivery_duration_min <dbl> 37, 21, 55, 38, 23, 33, 37, 62, 76, 34, 45, NA, …
$ city                  <chr> "New York", "Dallas", "San Jose", "San Diego", "…
$ state                 <chr> "NY", "TX", "CA", "CA", "PA", "CA", "TX", "NY", …
$ latitude              <dbl> 40.7128, 32.7767, 37.3382, 32.7157, 39.9526, 32.…
$ longitude             <dbl> -74.0060, -96.7970, -121.8863, -117.1611, -75.16…
$ restaurant            <chr> "Pasta Place", "Taco Town", "Burger Haven", "Sea…
$ is_canceled           <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE,…
$ cancel_reason         <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, "Oth…
$ customer_rating       <